# Система компьютерного зрения: распознавание геометрических фигур
## Перебор гиперпараметров нейронной сети — PyTorch (треугольник, круг, квадрат)

In [1]:
!pip install torch torchvision -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')
from PIL import Image
%matplotlib inline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используется устройство: {device}')

Используется устройство: cpu


In [3]:
!pip install gdown -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Загрузка датасета из облака
import gdown
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l3/hw_light.zip', None, quiet=True)

'hw_light.zip'

In [6]:
# Распаковываем архив
!unzip -q hw_light.zip

^C


In [5]:
# Путь к директории с базой
base_dir = 'hw_light'
x_train = []
y_train = []
img_height = 20
img_width  = 20

for patch in os.listdir(base_dir):
    for img_name in os.listdir(base_dir + '/' + patch):
        img_path = base_dir + '/' + patch + '/' + img_name
        # Загрузка изображения через PIL (аналог keras image.load_img)
        img = Image.open(img_path).convert('L').resize((img_width, img_height))
        x_train.append(np.array(img))

        if patch == '0':
            y_train.append(0)
        elif patch == '3':
            y_train.append(1)
        else:
            y_train.append(2)

x_train = np.array(x_train)
y_train = np.array(y_train)
print('Размер массива x_train', x_train.shape)
print('Размер массива y_train', y_train.shape)

Размер массива x_train (302, 20, 20)
Размер массива y_train (302,)


In [6]:
# Нормализация и разворот в вектор
x_train_flat = x_train.reshape(x_train.shape[0], -1).astype(np.float32) / 255.0
y_train_int  = y_train.astype(np.int64)

# Преобразование в тензоры PyTorch
X_tensor = torch.tensor(x_train_flat)
y_tensor = torch.tensor(y_train_int)

print('Форма X_tensor:', X_tensor.shape)
print('Форма y_tensor:', y_tensor.shape)

Форма X_tensor: torch.Size([302, 400])
Форма y_tensor: torch.Size([302])


In [7]:
# Определение модели на PyTorch

def build_model(input_dim, neurons, activation):
    """Создаёт Sequential-модель с одним скрытым слоем."""
    if activation == 'relu':
        act_fn = nn.ReLU()
    else:  # linear — нет нелинейности
        act_fn = nn.Identity()

    model = nn.Sequential(
        nn.Linear(input_dim, neurons),
        act_fn,
        nn.Linear(neurons, 3)   # 3 класса, softmax внутри CrossEntropyLoss
    )
    return model


def train_model(model, loader, epochs=10):
    """Обучает модель и возвращает финальную точность."""
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters())

    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

    # Оценка точности на всём тренировочном наборе
    model.eval()
    with torch.no_grad():
        outputs = model(X_tensor.to(device))
        preds   = torch.argmax(outputs, dim=1).cpu()
        acc     = (preds == y_tensor).float().mean().item()
    return acc

In [8]:
# ПЕРЕБОР ГИПЕРПАРАМЕТРОВ: 18 комбинаций
# Нейроны: [10, 100, 5000]
# Активации: ['relu', 'linear']
# Batch sizes: [10, 100, 1000]

neurons_list = [10, 100, 5000]
activations  = ['relu', 'linear']
batch_sizes  = [10, 100, 1000]

input_dim = X_tensor.shape[1]  # 400
epochs    = 10
results   = []
dataset   = TensorDataset(X_tensor, y_tensor)

combo_num = 0
for neurons in neurons_list:
    for activation in activations:
        for batch_size in batch_sizes:
            combo_num += 1
            print(f'[{combo_num}/18] Нейронов={neurons}, Активация={activation}, Batch={batch_size}', end=' ... ')

            loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
            model  = build_model(input_dim, neurons, activation)
            acc    = train_model(model, loader, epochs=epochs)

            print(f'Точность: {acc:.4f}')
            results.append({
                'Нейронов':   neurons,
                'Активация':  activation,
                'Batch size': batch_size,
                'Точность':   round(acc, 4)
            })

print('\n комбинаций протестированы')

[1/18] Нейронов=10, Активация=relu, Batch=10 ... Точность: 0.7848
[2/18] Нейронов=10, Активация=relu, Batch=100 ... Точность: 0.6490
[3/18] Нейронов=10, Активация=relu, Batch=1000 ... Точность: 0.6424
[4/18] Нейронов=10, Активация=linear, Batch=10 ... Точность: 0.7748
[5/18] Нейронов=10, Активация=linear, Batch=100 ... Точность: 0.6093
[6/18] Нейронов=10, Активация=linear, Batch=1000 ... Точность: 0.7483
[7/18] Нейронов=100, Активация=relu, Batch=10 ... Точность: 0.8874
[8/18] Нейронов=100, Активация=relu, Batch=100 ... Точность: 0.7185
[9/18] Нейронов=100, Активация=relu, Batch=1000 ... Точность: 0.7781
[10/18] Нейронов=100, Активация=linear, Batch=10 ... Точность: 0.8477
[11/18] Нейронов=100, Активация=linear, Batch=100 ... Точность: 0.7219
[12/18] Нейронов=100, Активация=linear, Batch=1000 ... Точность: 0.7318
[13/18] Нейронов=5000, Активация=relu, Batch=10 ... Точность: 0.9536
[14/18] Нейронов=5000, Активация=relu, Batch=100 ... Точность: 0.8377
[15/18] Нейронов=5000, Активация=rel

In [9]:
df = pd.DataFrame(results)
df.index = range(1, len(df) + 1)
df.index.name = '№'

print('\n Сравнительная таблица результатов')
print(df.to_string())

best = df.loc[df['Точность'].idxmax()]
print(f'\nЛучшая комбинация:')
print(best)


 Сравнительная таблица результатов
    Нейронов Активация  Batch size  Точность
№                                           
1         10      relu          10    0.7848
2         10      relu         100    0.6490
3         10      relu        1000    0.6424
4         10    linear          10    0.7748
5         10    linear         100    0.6093
6         10    linear        1000    0.7483
7        100      relu          10    0.8874
8        100      relu         100    0.7185
9        100      relu        1000    0.7781
10       100    linear          10    0.8477
11       100    linear         100    0.7219
12       100    linear        1000    0.7318
13      5000      relu          10    0.9536
14      5000      relu         100    0.8377
15      5000      relu        1000    0.5662
16      5000    linear          10    0.7285
17      5000    linear         100    0.6954
18      5000    linear        1000    0.6788

Лучшая комбинация:
Нейронов        5000
Активация       relu
Ba